# API-Football Data Collection

## Objective

The objective of this notebook is:

**collect the football data required for the project and store it locally by data category.**

No analytical groups are created here. SC Braga, Benfica, FC Porto, Sporting CP, and the rest of the league will only be filtered later when the JSON files are loaded into DataFrames.

The collection scope is:

- Primeira Liga as a whole for 2022/23, 2023/24, and 2024/25;
- SC Braga, Benfica, FC Porto, and Sporting CP across all competitions available in those seasons;
- detailed match data for the resulting unique fixtures.

The final raw-data structure is:

```text
data/
└── raw/
    └── api_football/
        ├── leagues/
        ├── teams/
        ├── fixtures/
        ├── events/
        ├── lineups/
        ├── statistics/
        └── players/
```

The important principle is:

**collect first, filter later.**


## 1. Imports


In [17]:
import os
import json
import math
import time
from pathlib import Path

import requests
from dotenv import load_dotenv


## 2. Project folders


In [18]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "api_football"

LEAGUES_DIR = DATA_DIR / "leagues"
TEAMS_DIR = DATA_DIR / "teams"
FIXTURES_DIR = DATA_DIR / "fixtures"
EVENTS_DIR = DATA_DIR / "events"
LINEUPS_DIR = DATA_DIR / "lineups"
STATISTICS_DIR = DATA_DIR / "statistics"
PLAYERS_DIR = DATA_DIR / "players"

for folder in [
    LEAGUES_DIR,
    TEAMS_DIR,
    FIXTURES_DIR,
    EVENTS_DIR,
    LINEUPS_DIR,
    STATISTICS_DIR,
    PLAYERS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print(DATA_DIR)


/Users/nathansperinde/Desktop/Portifolio/Football-Analysis/data/raw/api_football


## 3. API configuration


In [19]:
load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.getenv("API_FOOTBALL_KEY")

if not API_KEY:
    raise ValueError("API_FOOTBALL_KEY was not found.")

BASE_URL = "https://v3.football.api-sports.io"

HEADERS = {
    "x-apisports-key": API_KEY
}

print("API key loaded successfully.")


API key loaded successfully.


## 4. Simple helper functions

Only three small functions are needed:

- `save_json()` saves data;
- `load_json()` loads data already stored locally;
- `get_data()` calls the API.

The notebook checks for final files before repeating a full collection stage.


In [20]:
def save_json(data, path):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2
        )


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def get_data(endpoint, params):
    url = f"{BASE_URL}/{endpoint}"

    response = requests.get(
        url,
        headers=HEADERS,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    if data.get("errors"):
        raise RuntimeError(data["errors"])

    print(
        f"/{endpoint} | "
        f"{params} | "
        f"results: {data.get('results')}"
    )

    time.sleep(7)

    return data


## 5. Study scope

The API uses the starting year of the football season.

Therefore:

- `2022` = 2022/23;
- `2023` = 2023/24;
- `2024` = 2024/25.

The four target clubs are only used here to make sure their matches from **all competitions** enter the global fixture dataset. They are not stored in separate analytical datasets.


In [21]:
SEASONS = [2022, 2023, 2024]

TARGET_TEAMS = [
    "SC Braga",
    "Benfica",
    "FC Porto",
    "Sporting CP"
]


## 6. Leagues

We first retrieve Portuguese competitions and identify the Primeira Liga ID.

The response is saved locally so this request does not need to be repeated in future sessions.


In [22]:
LEAGUES_FILE = LEAGUES_DIR / "leagues.json"

if LEAGUES_FILE.exists():

    leagues_data = load_json(LEAGUES_FILE)
    print("Loaded leagues from local file.")

else:

    leagues_data = get_data(
        endpoint="leagues",
        params={
            "country": "Portugal",
            "season": 2024
        }
    )

    save_json(
        leagues_data,
        LEAGUES_FILE
    )

    print(f"Saved: {LEAGUES_FILE}")


Loaded leagues from local file.


In [23]:
primeira_liga_id = None

for item in leagues_data["response"]:

    if item["league"]["name"] == "Primeira Liga":
        primeira_liga_id = item["league"]["id"]
        break

if primeira_liga_id is None:
    raise ValueError("Primeira Liga was not found.")

print(f"Primeira Liga ID: {primeira_liga_id}")


Primeira Liga ID: 94


## 7. Teams

All Primeira Liga teams from the three seasons are stored together in a single file:

`data/raw/api_football/teams/teams.json`

The season is added to each record so promoted and relegated clubs remain distinguishable later.


In [24]:
TEAMS_FILE = TEAMS_DIR / "teams.json"

if TEAMS_FILE.exists():

    teams = load_json(TEAMS_FILE)
    print("Loaded teams from local file.")

else:

    teams = []

    for season in SEASONS:

        data = get_data(
            endpoint="teams",
            params={
                "league": primeira_liga_id,
                "season": season
            }
        )

        for item in data["response"]:

            item["season"] = season
            teams.append(item)

    save_json(
        teams,
        TEAMS_FILE
    )

    print(f"Saved: {TEAMS_FILE}")


Loaded teams from local file.


### Find the IDs of the four target clubs

The IDs are obtained from the data already collected above. No additional API request is needed.


In [25]:
team_ids = {}

for item in teams:

    team_name = item["team"]["name"]

    if team_name in TARGET_TEAMS:
        team_ids[team_name] = item["team"]["id"]

team_ids


{'Benfica': 211, 'FC Porto': 212, 'SC Braga': 217, 'Sporting CP': 228}

## 8. Fixtures

This is the most important collection stage.

Two types of API queries are used:

1. all Primeira Liga fixtures for each season;
2. all fixtures from each of the four target clubs for each season.

However, **the results are not kept separate**.

All returned matches are immediately joined, duplicate `fixture.id` values are removed, and the final result is stored as one file:

`data/raw/api_football/fixtures/fixtures.json`

This means later analysis can decide whether to filter:

- SC Braga;
- Benfica;
- FC Porto;
- Sporting CP;
- Big 3;
- rest of league;
- league average;
- European competitions;
- domestic cups;
- or any other group.

The raw collection does not make that decision.


In [26]:
FIXTURES_FILE = FIXTURES_DIR / "fixtures.json"

if FIXTURES_FILE.exists():

    fixtures = load_json(FIXTURES_FILE)
    print("Loaded fixtures from local file.")

else:

    all_fixtures = []

    for season in SEASONS:

        data = get_data(
            endpoint="fixtures",
            params={
                "league": primeira_liga_id,
                "season": season
            }
        )

        all_fixtures.extend(
            data["response"]
        )

    for team_name, team_id in team_ids.items():

        for season in SEASONS:

            data = get_data(
                endpoint="fixtures",
                params={
                    "team": team_id,
                    "season": season
                }
            )

            all_fixtures.extend(
                data["response"]
            )

    fixtures_by_id = {}

    for fixture in all_fixtures:

        fixture_id = fixture["fixture"]["id"]
        fixtures_by_id[fixture_id] = fixture

    fixtures = list(
        fixtures_by_id.values()
    )

    save_json(
        fixtures,
        FIXTURES_FILE
    )

    print(f"Saved: {FIXTURES_FILE}")
    print(f"Unique fixtures: {len(fixtures)}")


Loaded fixtures from local file.


## 9. Competitions found in the fixtures

The four clubs may have played in domestic cups and UEFA competitions.

There is no need to make another API request just to discover those competitions because the fixture response already contains the league/competition information.

The unique competition metadata are therefore extracted locally and stored with the league data.


In [27]:
COMPETITIONS_FILE = LEAGUES_DIR / "competitions.json"

competitions_by_id = {}

for item in fixtures:

    league = item["league"]

    competitions_by_id[
        league["id"]
    ] = league

competitions = list(
    competitions_by_id.values()
)

save_json(
    competitions,
    COMPETITIONS_FILE
)

print(f"Saved: {COMPETITIONS_FILE}")
print(f"Competitions found: {len(competitions)}")


Saved: /Users/nathansperinde/Desktop/Portifolio/Football-Analysis/data/raw/api_football/leagues/competitions.json
Competitions found: 8


## 10. Detailed match data

The basic fixture dataset contains match information such as teams, competition, date, score, and status. Additional match-level data are retrieved separately using each fixture ID.

These detailed responses can contain:

- events;
- lineups;
- team match statistics;
- player match statistics.

Because the API-Football Free plan does not support requesting multiple fixture IDs through the `ids` parameter, detailed data are collected one fixture at a time using:

`/fixtures?id=<fixture_id>`

Each response is stored separately inside:

`data/raw/api_football/fixtures/details/`

Only fixtures that have already been played are included in this stage. Existing files are preserved and reused, preventing the same fixture from consuming another API request in future executions.

In [28]:
DETAILS_DIR = (
    FIXTURES_DIR / "details"
)

DETAILS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

fixture_ids = [
    item["fixture"]["id"]
    for item in fixtures
    if item["fixture"]["status"]["short"]
    not in ["NS", "TBD"]
]

print(
    f"Fixtures with possible match data: "
    f"{len(fixture_ids)}"
)

Fixtures with possible match data: 1222


### Run detailed collection

This is the stage that consumes the largest number of API requests because each fixture requires an individual request.

`RUN_DETAILS` controls whether the collection is executed, while `MAX_NEW_REQUESTS` limits the number of new requests made during a single execution.

Before requesting a fixture, the notebook checks whether its JSON file already exists locally. Existing fixtures are skipped and do not count towards the request limit.

This allows the collection to be completed progressively across multiple days without downloading the same match data again.

In [29]:
RUN_DETAILS = True

MAX_NEW_REQUESTS = 90

if RUN_DETAILS:

    new_requests = 0

    for fixture_id in fixture_ids:

        fixture_file = (
            DETAILS_DIR
            / f"fixture_{fixture_id}.json"
        )

        if fixture_file.exists():

            print(
                f"Fixture {fixture_id} "
                f"already exists."
            )

            continue

        if new_requests >= MAX_NEW_REQUESTS:

            print(
                "Maximum number of new "
                "requests reached."
            )

            break

        data = get_data(
            endpoint="fixtures",
            params={
                "id": fixture_id
            }
        )

        save_json(
            data["response"],
            fixture_file
        )

        new_requests += 1

        print(
            f"Saved fixture {fixture_id} | "
            f"New requests: {new_requests}"
        )

else:

    print(
        "Detailed collection disabled. "
        "Set RUN_DETAILS = True when ready."
    )

Fixture 898605 already exists.
Fixture 898604 already exists.
Fixture 898608 already exists.
Fixture 898612 already exists.
Fixture 898610 already exists.
Fixture 898609 already exists.
Fixture 898606 already exists.
Fixture 898611 already exists.
Fixture 898607 already exists.
Fixture 898617 already exists.
Fixture 898620 already exists.
Fixture 898615 already exists.
Fixture 898618 already exists.
Fixture 898614 already exists.
Fixture 898613 already exists.
Fixture 898616 already exists.
Fixture 898621 already exists.
Fixture 898619 already exists.
Fixture 898629 already exists.
Fixture 898625 already exists.
Fixture 898628 already exists.
Fixture 898626 already exists.
Fixture 898622 already exists.
Fixture 898627 already exists.
Fixture 898624 already exists.
Fixture 898623 already exists.
Fixture 898631 already exists.
Fixture 898636 already exists.
Fixture 898637 already exists.
Fixture 898634 already exists.
Fixture 898638 already exists.
Fixture 898639 already exists.
Fixture 

## 11. Load detailed fixture data

This stage does not call the API.

It reads all individual fixture files already stored in:

`data/raw/api_football/fixtures/details/`

The files are combined into a single list called `detailed_fixtures`, which is then used to extract events, lineups, team statistics, and player statistics in the following stage.

Because the data are loaded entirely from local files, this step can be executed repeatedly without consuming API requests.

In [30]:
detail_files = sorted(
    DETAILS_DIR.glob(
        "fixture_*.json"
    )
)

detailed_fixtures = []

for file in detail_files:

    fixture_data = load_json(file)

    detailed_fixtures.extend(
        fixture_data
    )

print(
    f"Detailed fixtures loaded: "
    f"{len(detailed_fixtures)}"
)

Detailed fixtures loaded: 1222


## 12. Separate detailed data by category

The detailed fixture response already contains several types of information.

Instead of making more API requests, these nested objects are separated locally into the corresponding folders.

No analytical filtering is performed.


In [31]:
events = []
lineups = []
statistics = []
player_statistics = []

for item in detailed_fixtures:

    fixture_id = item["fixture"]["id"]

    for event in item.get(
        "events",
        []
    ):

        event["fixture_id"] = fixture_id
        events.append(event)

    for lineup in item.get(
        "lineups",
        []
    ):

        lineup["fixture_id"] = fixture_id
        lineups.append(lineup)

    for statistic in item.get(
        "statistics",
        []
    ):

        statistic["fixture_id"] = fixture_id
        statistics.append(statistic)

    for player_data in item.get(
        "players",
        []
    ):

        player_data["fixture_id"] = fixture_id
        player_statistics.append(
            player_data
        )


In [32]:
EVENTS_FILE = EVENTS_DIR / "events.json"
LINEUPS_FILE = LINEUPS_DIR / "lineups.json"
STATISTICS_FILE = STATISTICS_DIR / "statistics.json"
PLAYERS_FILE = (
    PLAYERS_DIR
    / "player_match_statistics.json"
)

save_json(
    events,
    EVENTS_FILE
)

save_json(
    lineups,
    LINEUPS_FILE
)

save_json(
    statistics,
    STATISTICS_FILE
)

save_json(
    player_statistics,
    PLAYERS_FILE
)

print(f"Events: {len(events)}")
print(f"Lineups: {len(lineups)}")
print(f"Team statistics: {len(statistics)}")
print(
    "Player match-statistic blocks: "
    f"{len(player_statistics)}"
)


Events: 21214
Lineups: 2356
Team statistics: 2238
Player match-statistic blocks: 2184


## 13. Final structure

After collection, the important files are:

```text
data/
└── raw/
    └── api_football/
        ├── leagues/
        │   ├── leagues.json
        │   └── competitions.json
        │
        ├── teams/
        │   └── teams.json
        │
        ├── fixtures/
        │   ├── fixtures.json
        │   └── detail_batches/
        │
        ├── events/
        │   └── events.json
        │
        ├── lineups/
        │   └── lineups.json
        │
        ├── statistics/
        │   └── statistics.json
        │
        └── players/
            └── player_match_statistics.json
```
